
# pandas Learning Notebook

This notebook is a practical guide to the **pandas** Python library.

It covers:
- Jupyter setup
- `Series` and `DataFrame`
- loading and saving data
- indexing and selection
- filtering and sorting
- missing values
- adding, changing, and dropping columns
- string methods
- grouping and aggregation
- merging and joining
- reshaping with pivot tables and melt
- dates and time series
- applying functions
- useful summary and inspection helpers

**Note:** pandas is a large library, so this notebook focuses on the **most commonly used methods, functions, and features** rather than every single API entry.



## 1) Jupyter setup

Install pandas and common tools:

```bash
pip install pandas numpy matplotlib jupyter
jupyter notebook
```

or with conda:

```bash
conda install pandas numpy matplotlib jupyter
jupyter notebook
```


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)


## 2) Create a simple dataset

In [ ]:

data = {
    "student_id": [101, 102, 103, 104, 105, 106],
    "name": ["Fiona", "Alex", "Sam", "Jordan", "Maya", "Noah"],
    "course": ["Python", "Python", "SQL", "ML", "SQL", "ML"],
    "score": [92, 85, 78, 95, np.nan, 88],
    "hours_studied": [12, 9, 7, 15, 6, 11],
    "passed": [True, True, False, True, False, True],
    "city": ["Toronto", "Montreal", "Toronto", "Vancouver", "Toronto", "Montreal"],
    "exam_date": pd.to_datetime(["2026-03-01", "2026-03-01", "2026-03-02", "2026-03-02", "2026-03-03", "2026-03-03"])
}

df = pd.DataFrame(data)
df


In [ ]:

Path("pandas_data").mkdir(exist_ok=True)
df.to_csv("pandas_data/students.csv", index=False)
print("Saved sample dataset to pandas_data/students.csv")


## 3) Series vs DataFrame

In [ ]:

scores = df["score"]   # Series
print("Type of df['score']:", type(scores))
print(scores)


In [ ]:

print("Type of df:", type(df))
print(df)


## 4) Basic inspection

In [ ]:

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Index:", df.index)
print("Dtypes:\n", df.dtypes)


In [ ]:

df.head()


In [ ]:

df.tail(3)


In [ ]:

df.info()


In [ ]:

df.describe(include="all")


## 5) Load data from files

In [ ]:

loaded_csv = pd.read_csv("pandas_data/students.csv")
loaded_csv.head()


In [ ]:

loaded_csv["exam_date"] = pd.to_datetime(loaded_csv["exam_date"])
loaded_csv.dtypes


## 6) Select columns

In [ ]:

df["name"]


In [ ]:

df[["name", "course", "score"]]


## 7) Row selection with `.loc` and `.iloc`

In [ ]:

df.loc[0]


In [ ]:

df.loc[0:2, ["name", "score", "city"]]


In [ ]:

df.iloc[0]


In [ ]:

df.iloc[0:3, 1:4]


## 8) Boolean filtering

In [ ]:

df[df["score"] >= 90]


In [ ]:

df[(df["course"] == "Python") & (df["passed"] == True)]


In [ ]:

df[df["city"].isin(["Toronto", "Montreal"])]


In [ ]:

df[df["name"].str.contains("a", case=False)]


## 9) Sorting

In [ ]:

df.sort_values("score", ascending=False)


In [ ]:

df.sort_values(["course", "score"], ascending=[True, False])


## 10) Missing values

In [ ]:

df.isna()


In [ ]:

df.isna().sum()


In [ ]:

df["score"].fillna(df["score"].mean())


In [ ]:

df_filled = df.copy()
df_filled["score"] = df_filled["score"].fillna(df_filled["score"].mean())
df_filled


In [ ]:

df.dropna()


## 11) Add, change, and drop columns

In [ ]:

df2 = df.copy()
df2["score_filled"] = df2["score"].fillna(df2["score"].mean())
df2["score_percent"] = df2["score_filled"] / 100
df2["study_level"] = np.where(df2["hours_studied"] >= 10, "High", "Low")
df2


In [ ]:

df2["name_upper"] = df2["name"].str.upper()
df2


In [ ]:

df2 = df2.drop(columns=["name_upper"])
df2.head()


## 12) Rename columns

In [ ]:

df2.rename(columns={"hours_studied": "hours"})


## 13) String methods

In [ ]:

df["name"].str.lower()


In [ ]:

df["course"].str.replace("ML", "Machine Learning")


In [ ]:

df["email"] = df["name"].str.lower() + "@example.com"
df[["name", "email"]]


## 14) Value counts and unique values

In [ ]:

df["course"].value_counts()


In [ ]:

print("Unique cities:", df["city"].unique())
print("Number of unique cities:", df["city"].nunique())


## 15) Apply functions

In [ ]:

df["hours_studied"].apply(lambda x: x * 2)


In [ ]:

def grade_label(score):
    if pd.isna(score):
        return "Missing"
    elif score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "D"

df["grade"] = df["score"].apply(grade_label)
df[["name", "score", "grade"]]


## 16) Aggregations

In [ ]:

print("Mean score:", df["score"].mean())
print("Median score:", df["score"].median())
print("Min score:", df["score"].min())
print("Max score:", df["score"].max())
print("Sum of study hours:", df["hours_studied"].sum())


## 17) GroupBy

In [ ]:

df.groupby("course")["score"].mean()


In [ ]:

df.groupby("city")[["score", "hours_studied"]].mean(numeric_only=True)


In [ ]:

df.groupby(["course", "city"]).agg({
    "score": ["mean", "max"],
    "hours_studied": ["sum", "mean"]
})


## 18) Transform

In [ ]:

df["course_avg_score"] = df.groupby("course")["score"].transform("mean")
df[["name", "course", "score", "course_avg_score"]]


## 19) Merge and join

In [ ]:

attendance = pd.DataFrame({
    "student_id": [101, 102, 103, 104, 107],
    "attendance_rate": [0.95, 0.88, 0.76, 0.99, 0.80]
})

attendance


In [ ]:

merged = pd.merge(df, attendance, on="student_id", how="left")
merged


In [ ]:

left = pd.DataFrame({"id": [1, 2, 3], "value_left": ["A", "B", "C"]})
right = pd.DataFrame({"id": [2, 3, 4], "value_right": ["X", "Y", "Z"]})

print("Inner join:")
display(pd.merge(left, right, on="id", how="inner"))

print("Left join:")
display(pd.merge(left, right, on="id", how="left"))

print("Outer join:")
display(pd.merge(left, right, on="id", how="outer"))


## 20) Concatenate

In [ ]:

part1 = df.iloc[:3]
part2 = df.iloc[3:]

pd.concat([part1, part2], axis=0)


## 21) Pivot tables

In [ ]:

pd.pivot_table(
    df,
    values="hours_studied",
    index="course",
    columns="city",
    aggfunc="mean"
)


## 22) Melt / long format

In [ ]:

wide_df = pd.DataFrame({
    "student": ["Fiona", "Alex"],
    "math": [90, 80],
    "science": [95, 85]
})

wide_df


In [ ]:

pd.melt(wide_df, id_vars="student", var_name="subject", value_name="score")


## 23) Dates and time series

In [ ]:

dates = pd.date_range(start="2026-03-01", periods=6, freq="D")
ts = pd.Series([10, 12, 9, 15, 14, 18], index=dates)
ts


In [ ]:

print("Month:")
print(df["exam_date"].dt.month)

print("\nDay name:")
print(df["exam_date"].dt.day_name())


In [ ]:

ts.rolling(3).mean()


## 24) Duplicates

In [ ]:

dup_df = pd.DataFrame({
    "name": ["A", "A", "B", "C", "C"],
    "score": [1, 1, 2, 3, 3]
})

print("Duplicates:")
display(dup_df.duplicated())

print("\nWithout duplicates:")
display(dup_df.drop_duplicates())


## 25) Index operations

In [ ]:

indexed = df.set_index("student_id")
indexed


In [ ]:

indexed.loc[101]


In [ ]:

indexed.reset_index().head()


## 26) Query method

In [ ]:

df.query("hours_studied >= 10 and passed == True")


## 27) Sampling

In [ ]:

df.sample(3, random_state=42)


## 28) Export data

In [ ]:

df.to_csv("pandas_data/students_export.csv", index=False)
df.to_json("pandas_data/students_export.json", orient="records", indent=2)

print("Saved:")
print("- pandas_data/students_export.csv")
print("- pandas_data/students_export.json")


## 29) Simple plotting with pandas

In [ ]:

df["hours_studied"].plot(kind="bar", figsize=(8, 4), title="Hours Studied")
plt.xlabel("Row index")
plt.ylabel("Hours")
plt.show()


In [ ]:

df["score"].plot(kind="hist", bins=5, figsize=(8, 4), title="Score Distribution")
plt.xlabel("Score")
plt.show()


## 30) Common pandas objects and functions

In [ ]:

pandas_reference = {
    "Core objects": ["Series", "DataFrame", "Index"],
    "Loading/saving": ["read_csv", "read_excel", "read_json", "to_csv", "to_excel", "to_json"],
    "Inspection": ["head", "tail", "info", "describe", "shape", "dtypes", "columns"],
    "Selection": ["loc", "iloc", "query", "set_index", "reset_index"],
    "Cleaning": ["isna", "fillna", "dropna", "drop_duplicates", "rename", "astype"],
    "Sorting/filtering": ["sort_values", "sort_index", "isin", "value_counts", "unique", "nunique"],
    "Group/aggregate": ["groupby", "agg", "transform", "mean", "sum", "count"],
    "Combine/reshape": ["merge", "join", "concat", "pivot_table", "melt"],
    "Time series": ["to_datetime", "date_range", "rolling"],
    "Strings": ["str.lower", "str.upper", "str.contains", "str.replace"],
    "Apply": ["apply", "map"]
}

for category, funcs in pandas_reference.items():
    print(f"\n{category}:")
    print(", ".join(funcs))


## 31) Inspect public names

In [ ]:

public_pd_names = [name for name in dir(pd) if not name.startswith("_")]
print("Some public names in pandas:")
print(public_pd_names[:200])
print("\nTotal shown:", len(public_pd_names))



## 32) Practice exercises

1. Create a DataFrame with 5 rows and 4 columns.
2. Select one column as a Series.
3. Filter rows where a numeric column is greater than a value.
4. Fill missing values in one column with the column mean.
5. Group by a category column and compute the mean.
6. Merge two DataFrames on a common key.
7. Convert a wide DataFrame to long format with `melt`.
8. Parse a date column with `pd.to_datetime`.
9. Save a DataFrame to CSV and JSON.
10. Plot a histogram of one numeric column.



## 33) Summary

You now have a notebook that introduces the main pandas workflow:

- creating and inspecting `Series` and `DataFrame`
- selecting rows and columns
- filtering and sorting
- handling missing values
- adding and transforming columns
- grouping and aggregating
- merging and reshaping data
- working with dates and time series
- saving and plotting data

This is the core path used in many data analysis, ML, and automation projects.
